In [1]:
import os
import random
import time
from dataclasses import dataclass
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import tyro
from torch.utils.tensorboard import SummaryWriter
from cleanrl_utils.buffers import ReplayBuffer

In [2]:
__file__="d:\\Users\\xjj\\AppData\\Local\\Programs\\python_ws\\cleanrl-master\\cleanrl\\sac_continuous_action.py"

In [3]:
@dataclass
class Args:
    exp_name: str = os.path.basename(__file__)[: -len(".py")]
    """the name of this experiment"""
    seed: int = 1
    """seed of the experiment"""
    torch_deterministic: bool = True
    """if toggled, `torch.backends.cudnn.deterministic=False`"""
    cuda: bool = True
    """if toggled, cuda will be enabled by default"""
    track: bool = False
    """if toggled, this experiment will be tracked with Weights and Biases"""
    wandb_project_name: str = "cleanRL"
    """the wandb's project name"""
    wandb_entity: str = None
    """the entity (team) of wandb's project"""
    capture_video: bool = True
    """whether to capture videos of the agent performances (check out `videos` folder)"""

    # Algorithm specific arguments
    env_id: str = "Hopper-v4"
    """the environment id of the task"""
    total_timesteps: int = 1000000
    """total timesteps of the experiments"""
    num_envs: int = 1
    """the number of parallel game environments"""
    buffer_size: int = int(1e6)
    """the replay memory buffer size"""
    gamma: float = 0.99
    """the discount factor gamma"""
    tau: float = 0.005
    """target smoothing coefficient (default: 0.005)"""
    batch_size: int = 256
    """the batch size of sample from the reply memory"""
    learning_starts: int = 5e3
    """timestep to start learning"""
    policy_lr: float = 3e-4
    """the learning rate of the policy network optimizer"""
    q_lr: float = 1e-3
    """the learning rate of the Q network network optimizer"""
    policy_frequency: int = 2
    """the frequency of training policy (delayed)"""
    target_network_frequency: int = 1  # Denis Yarats' implementation delays this by 2.
    """the frequency of updates for the target nerworks"""
    alpha: float = 0.2
    """Entropy regularization coefficient."""
    autotune: bool = True
    """automatic tuning of the entropy coefficient"""

In [4]:
def make_env(env_id, seed, idx, capture_video, run_name):
    def thunk():
        if capture_video and idx == 0:#if (capture_video)和(idx==0)
            env = gym.make(env_id, render_mode="rgb_array")
            env = gym.wrappers.RecordVideo(env, f"videos/{run_name}")
        else:
            env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        env.action_space.seed(seed)
        return env

    return thunk

In [5]:
class SoftQNetwork(nn.Module):
    def __init__(self, env):
        super().__init__()
        self.fc1 = nn.Linear(
            np.array(env.single_observation_space.shape).prod() + np.prod(env.single_action_space.shape),
            256,
        )
        self.fc2 = nn.Linear(256, 256)
        self.fc3 = nn.Linear(256, 1)

    def forward(self, x, a):
        x = torch.cat([x, a], 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [6]:
LOG_STD_MAX = 2
LOG_STD_MIN = -5

In [7]:
class Actor(nn.Module):
    def __init__(self, env):
        super().__init__()
        self.fc1 = nn.Linear(np.array(env.single_observation_space.shape).prod(), 256)
        self.fc2 = nn.Linear(256, 256)
        self.fc_mean = nn.Linear(256, np.prod(env.single_action_space.shape))
        self.fc_logstd = nn.Linear(256, np.prod(env.single_action_space.shape))
        # action rescaling
        self.register_buffer(
            "action_scale",
            torch.tensor(
                (env.single_action_space.high - env.single_action_space.low) / 2.0,
                dtype=torch.float32,
            ),
        )
        self.register_buffer(
            "action_bias",
            torch.tensor(
                (env.single_action_space.high + env.single_action_space.low) / 2.0,
                dtype=torch.float32,
            ),
        )

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        mean = self.fc_mean(x)
        log_std = self.fc_logstd(x)
        log_std = torch.tanh(log_std)
        log_std = LOG_STD_MIN + 0.5 * (LOG_STD_MAX - LOG_STD_MIN) * (log_std + 1)  # From SpinUp / Denis Yarats

        return mean, log_std

    def get_action(self, x):
        mean, log_std = self(x)
        std = log_std.exp()
        normal = torch.distributions.Normal(mean, std)
        x_t = normal.rsample()  # for reparameterization trick (mean + std * N(0,1))
        y_t = torch.tanh(x_t)
        action = y_t * self.action_scale + self.action_bias
        log_prob = normal.log_prob(x_t)
        # Enforcing Action Bound
        log_prob -= torch.log(self.action_scale * (1 - y_t.pow(2)) + 1e-6)
        log_prob = log_prob.sum(1, keepdim=True)
        mean = torch.tanh(mean) * self.action_scale + self.action_bias
        return action, log_prob, mean

In [8]:
if __name__ == "__main__":

    args = Args()
    run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"
    if args.track:
        import wandb

        wandb.init(
            project=args.wandb_project_name,
            entity=args.wandb_entity,
            sync_tensorboard=True,
            config=vars(args),
            name=run_name,
            monitor_gym=True,
            save_code=True,
        )
    writer = SummaryWriter(f"runs/{run_name}")
    writer.add_text(
        "hyperparameters",
        "|param|value|\n|-|-|\n%s" % ("\n".join([f"|{key}|{value}|" for key, value in vars(args).items()])),
    )

    # TRY NOT TO MODIFY: seeding
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    torch.backends.cudnn.deterministic = args.torch_deterministic

    device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")

    # env setup
    envs = gym.vector.SyncVectorEnv(
        [make_env(args.env_id, args.seed + i, i, args.capture_video, run_name) for i in range(args.num_envs)]
    )
    assert isinstance(envs.single_action_space, gym.spaces.Box), "only continuous action space is supported"

    max_action = float(envs.single_action_space.high[0])

    actor = Actor(envs).to(device)
    qf1 = SoftQNetwork(envs).to(device)
    qf2 = SoftQNetwork(envs).to(device)
    qf1_target = SoftQNetwork(envs).to(device)
    qf2_target = SoftQNetwork(envs).to(device)
    qf1_target.load_state_dict(qf1.state_dict())
    qf2_target.load_state_dict(qf2.state_dict())
    q_optimizer = optim.Adam(list(qf1.parameters()) + list(qf2.parameters()), lr=args.q_lr)
    actor_optimizer = optim.Adam(list(actor.parameters()), lr=args.policy_lr)

    # Automatic entropy tuning
    if args.autotune:
        target_entropy = -torch.prod(torch.Tensor(envs.single_action_space.shape).to(device)).item()
        log_alpha = torch.zeros(1, requires_grad=True, device=device)
        alpha = log_alpha.exp().item()
        a_optimizer = optim.Adam([log_alpha], lr=args.q_lr)
    else:
        alpha = args.alpha

    envs.single_observation_space.dtype = np.float32
    rb = ReplayBuffer(
        args.buffer_size,
        envs.single_observation_space,
        envs.single_action_space,
        device,
        n_envs=args.num_envs,
        handle_timeout_termination=False,
    )
    start_time = time.time()

    # TRY NOT TO MODIFY: start the game
    obs, _ = envs.reset(seed=args.seed)
    for global_step in range(args.total_timesteps):
        # ALGO LOGIC: put action logic here
        if global_step < args.learning_starts:
            actions = np.array([envs.single_action_space.sample() for _ in range(envs.num_envs)])
        else:
            actions, _, _ = actor.get_action(torch.Tensor(obs).to(device))
            actions = actions.detach().cpu().numpy()

        # TRY NOT TO MODIFY: execute the game and log data.
        next_obs, rewards, terminations, truncations, infos = envs.step(actions)

        # TRY NOT TO MODIFY: record rewards for plotting purposes
        if "final_info" in infos:
            for info in infos["final_info"]:
                if info is not None:
                    print(f"global_step={global_step}, episodic_return={info['episode']['r']}")
                    writer.add_scalar("charts/episodic_return", info["episode"]["r"], global_step)
                    writer.add_scalar("charts/episodic_length", info["episode"]["l"], global_step)
                    break

        # TRY NOT TO MODIFY: save data to reply buffer; handle `final_observation`
        real_next_obs = next_obs.copy()
        for idx, trunc in enumerate(truncations):
            if trunc:
                real_next_obs[idx] = infos["final_observation"][idx]
        rb.add(obs, real_next_obs, actions, rewards, terminations, infos)

        # TRY NOT TO MODIFY: CRUCIAL step easy to overlook
        obs = next_obs

        # ALGO LOGIC: training.
        if global_step > args.learning_starts:
            data = rb.sample(args.batch_size)
            with torch.no_grad():
                next_state_actions, next_state_log_pi, _ = actor.get_action(data.next_observations)
                qf1_next_target = qf1_target(data.next_observations, next_state_actions)
                qf2_next_target = qf2_target(data.next_observations, next_state_actions)
                min_qf_next_target = torch.min(qf1_next_target, qf2_next_target) - alpha * next_state_log_pi
                next_q_value = data.rewards.flatten() + (1 - data.dones.flatten()) * args.gamma * (min_qf_next_target).view(-1)

            qf1_a_values = qf1(data.observations, data.actions).view(-1)
            qf2_a_values = qf2(data.observations, data.actions).view(-1)
            qf1_loss = F.mse_loss(qf1_a_values, next_q_value)
            qf2_loss = F.mse_loss(qf2_a_values, next_q_value)
            qf_loss = qf1_loss + qf2_loss

            # optimize the model
            q_optimizer.zero_grad()
            qf_loss.backward()
            q_optimizer.step()

            if global_step % args.policy_frequency == 0:  # TD 3 Delayed update support
                for _ in range(
                    args.policy_frequency
                ):  # compensate for the delay by doing 'actor_update_interval' instead of 1
                    pi, log_pi, _ = actor.get_action(data.observations)
                    qf1_pi = qf1(data.observations, pi)
                    qf2_pi = qf2(data.observations, pi)
                    min_qf_pi = torch.min(qf1_pi, qf2_pi)
                    actor_loss = ((alpha * log_pi) - min_qf_pi).mean()

                    actor_optimizer.zero_grad()
                    actor_loss.backward()
                    actor_optimizer.step()

                    if args.autotune:
                        with torch.no_grad():
                            _, log_pi, _ = actor.get_action(data.observations)
                        alpha_loss = (-log_alpha.exp() * (log_pi + target_entropy)).mean()

                        a_optimizer.zero_grad()
                        alpha_loss.backward()
                        a_optimizer.step()
                        alpha = log_alpha.exp().item()

            # update the target networks
            if global_step % args.target_network_frequency == 0:
                for param, target_param in zip(qf1.parameters(), qf1_target.parameters()):
                    target_param.data.copy_(args.tau * param.data + (1 - args.tau) * target_param.data)
                for param, target_param in zip(qf2.parameters(), qf2_target.parameters()):
                    target_param.data.copy_(args.tau * param.data + (1 - args.tau) * target_param.data)

            if global_step % 100 == 0:
                writer.add_scalar("losses/qf1_values", qf1_a_values.mean().item(), global_step)
                writer.add_scalar("losses/qf2_values", qf2_a_values.mean().item(), global_step)
                writer.add_scalar("losses/qf1_loss", qf1_loss.item(), global_step)
                writer.add_scalar("losses/qf2_loss", qf2_loss.item(), global_step)
                writer.add_scalar("losses/qf_loss", qf_loss.item() / 2.0, global_step)
                writer.add_scalar("losses/actor_loss", actor_loss.item(), global_step)
                writer.add_scalar("losses/alpha", alpha, global_step)
                print("SPS:", int(global_step / (time.time() - start_time)))
                writer.add_scalar(
                    "charts/SPS",
                    int(global_step / (time.time() - start_time)),
                    global_step,
                )
                if args.autotune:
                    writer.add_scalar("losses/alpha_loss", alpha_loss.item(), global_step)

    envs.close()
    writer.close()

D:\Users\xjj\anaconda3\envs\pytorch\lib\site-packages\gymnasium\utils\passive_env_checker.py:135: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be <class 'numpy.float32'>, actual type: float64
  logger.warn(
D:\Users\xjj\anaconda3\envs\pytorch\lib\site-packages\gymnasium\utils\passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
D:\Users\xjj\anaconda3\envs\pytorch\lib\site-packages\gymnasium\utils\passive_env_checker.py:135: UserWarning: WARN: The obs returned by the `step()` method was expecting numpy array dtype to be <class 'numpy.float32'>, actual type: float64
  logger.warn(
D:\Users\xjj\anaconda3\envs\pytorch\lib\site-packages\gymnasium\utils\passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within t

Moviepy - Building video D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-0.mp4.
Moviepy - Writing video D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-0.mp4



Moviepy - Done !
Moviepy - video ready D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-0.mp4
global_step=12, episodic_return=[10.137043]
Moviepy - Building video D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-1.mp4.
Moviepy - Writing video D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-1.mp4



Moviepy - Done !
Moviepy - video ready D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-1.mp4
global_step=33, episodic_return=[24.455343]
global_step=77, episodic_return=[31.36806]
global_step=97, episodic_return=[3.940759]
global_step=113, episodic_return=[10.493657]
global_step=145, episodic_return=[26.426783]
global_step=167, episodic_return=[9.872292]
global_step=208, episodic_return=[31.882452]
Moviepy - Building video D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-8.mp4.
Moviepy - Writing video D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-8.mp4



Moviepy - Done !
Moviepy - video ready D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-8.mp4
global_step=224, episodic_return=[16.432081]
global_step=243, episodic_return=[19.854567]
global_step=258, episodic_return=[12.445809]
global_step=277, episodic_return=[18.716702]
global_step=288, episodic_return=[7.640363]
global_step=328, episodic_return=[36.74399]
global_step=383, episodic_return=[49.284042]
global_step=396, episodic_return=[11.52337]
global_step=412, episodic_return=[13.856421]
global_step=426, episodic_return=[12.927986]
global_step=446, episodic_return=[16.879007]
global_step=464, episodic_return=[10.929363]
global_step=487, episodic_return=[17.046625]
global_step=509, episodic_return=[21.48278]
global_step=524, episodic_return=[10.287947]
global_step=538, episodic_return=[11.91034]
global_step=566, episodic_return=[15.661605]
global_step=578, episodic_return=[10.260446]
global_ste

...

Moviepy - Done !
Moviepy - video ready D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-64.mp4
global_step=1313, episodic_return=[7.8562207]
global_step=1331, episodic_return=[10.36663]
global_step=1351, episodic_return=[13.469153]
global_step=1380, episodic_return=[42.433372]
global_step=1408, episodic_return=[11.467732]
global_step=1421, episodic_return=[11.885272]
global_step=1438, episodic_return=[9.722403]
global_step=1456, episodic_return=[9.438201]
global_step=1470, episodic_return=[11.547862]
global_step=1494, episodic_return=[25.39861]
global_step=1531, episodic_return=[37.26705]
global_step=1542, episodic_return=[8.199955]
global_step=1556, episodic_return=[11.335049]
global_step=1580, episodic_return=[32.643375]
global_step=1601, episodic_return=[9.423699]
global_step=1615, episodic_return=[10.083982]
global_step=1641, episodic_return=[13.331328]
global_step=1664, episodic_return=[18.5

Moviepy - Done !
Moviepy - video ready D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-125.mp4
global_step=2775, episodic_return=[64.804504]
global_step=2803, episodic_return=[19.396564]
global_step=2831, episodic_return=[30.548983]
global_step=2840, episodic_return=[6.0371704]
global_step=2885, episodic_return=[65.01788]
global_step=2898, episodic_return=[9.12903]
global_step=2913, episodic_return=[13.905805]
global_step=2929, episodic_return=[13.027397]
global_step=2979, episodic_return=[43.758533]
global_step=3034, episodic_return=[77.63846]
global_step=3047, episodic_return=[10.7655325]
global_step=3065, episodic_return=[6.3251114]
global_step=3079, episodic_return=[9.265965]
global_step=3100, episodic_return=[13.634541]
global_step=3116, episodic_return=[12.509376]
global_step=3131, episodic_return=[12.068329]
global_step=3144, episodic_return=[10.899767]
global_step=3174, episodic_return=[

Moviepy - Done !
Moviepy - video ready D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-216.mp4
global_step=4733, episodic_return=[6.672391]
global_step=4752, episodic_return=[12.429984]
global_step=4761, episodic_return=[6.9110622]
global_step=4780, episodic_return=[15.171541]
global_step=4808, episodic_return=[20.739357]
global_step=4824, episodic_return=[12.86224]
global_step=4841, episodic_return=[13.085238]
global_step=4856, episodic_return=[12.285757]
global_step=4876, episodic_return=[17.414215]
global_step=4903, episodic_return=[29.09583]
global_step=4925, episodic_return=[12.067542]
global_step=4948, episodic_return=[23.91979]
global_step=4968, episodic_return=[17.156742]
global_step=4987, episodic_return=[14.252809]
global_step=5004, episodic_return=[9.414285]
global_step=5018, episodic_return=[9.753965]
global_step=5035, episodic_return=[11.013949]
global_step=5057, episodic_return=[11

Moviepy - Done !
Moviepy - video ready D:\Users\xjj\AppData\Local\Programs\python_ws\test\Jupyter_Notebook\videos\Hopper-v4__sac_continuous_action__1__1782804151\rl-video-episode-343.mp4
global_step=14237, episodic_return=[236.08263]
SPS: 66
global_step=14334, episodic_return=[225.19052]
SPS: 66
global_step=14427, episodic_return=[212.2352]
SPS: 66
global_step=14524, episodic_return=[224.93634]
SPS: 66
global_step=14620, episodic_return=[219.86807]
SPS: 65
global_step=14725, episodic_return=[243.04742]
SPS: 65
global_step=14819, episodic_return=[215.61028]
SPS: 65
global_step=14921, episodic_return=[235.98221]
SPS: 65
global_step=15022, episodic_return=[234.27025]
SPS: 65
global_step=15120, episodic_return=[230.52446]
SPS: 65
global_step=15215, episodic_return=[217.91565]
SPS: 64
global_step=15310, episodic_return=[217.10385]
SPS: 64
global_step=15411, episodic_return=[236.535]
SPS: 64
global_step=15508, episodic_return=[224.62076]
SPS: 64
global_step=15605, episodic_return=[225.3897]


KeyboardInterrupt: 

In [8]:
args = Args()

In [9]:
run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"

In [10]:
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")

In [12]:
envs = gym.vector.SyncVectorEnv(
    [make_env(args.env_id, args.seed + i, i, args.capture_video, run_name) for i in range(args.num_envs)]
)

In [13]:
max_action = float(envs.single_action_space.high[0])
max_action

1.0

In [14]:
actor = Actor(envs).to(device)
qf1 = SoftQNetwork(envs).to(device)
qf2 = SoftQNetwork(envs).to(device)
qf1_target = SoftQNetwork(envs).to(device)
qf2_target = SoftQNetwork(envs).to(device)
qf1_target.load_state_dict(qf1.state_dict())
qf2_target.load_state_dict(qf2.state_dict())
q_optimizer = optim.Adam(list(qf1.parameters()) + list(qf2.parameters()), lr=args.q_lr)
actor_optimizer = optim.Adam(list(actor.parameters()), lr=args.policy_lr)

In [15]:
if args.autotune:
    target_entropy = -torch.prod(torch.Tensor(envs.single_action_space.shape).to(device)).item()
    log_alpha = torch.zeros(1, requires_grad=True, device=device)
    alpha = log_alpha.exp().item()
    a_optimizer = optim.Adam([log_alpha], lr=args.q_lr)
else:
    alpha = args.alpha

In [16]:
envs.single_observation_space.dtype = np.float32
rb = ReplayBuffer(
    args.buffer_size,
    envs.single_observation_space,
    envs.single_action_space,
    device,
    n_envs=args.num_envs,
    handle_timeout_termination=False,
)
start_time = time.time()

In [17]:
obs, _ = envs.reset(seed=args.seed)

D:\Users\xjj\anaconda3\envs\pytorch\lib\site-packages\gymnasium\utils\passive_env_checker.py:135: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be <class 'numpy.float32'>, actual type: float64
  logger.warn(
D:\Users\xjj\anaconda3\envs\pytorch\lib\site-packages\gymnasium\utils\passive_env_checker.py:159: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


In [28]:
global_step = 9
actions = np.array([envs.single_action_space.sample() for _ in range(envs.num_envs)])
next_obs, rewards, terminations, truncations, infos = envs.step(actions)
real_next_obs = next_obs.copy()
rb.add(obs, real_next_obs, actions, rewards, terminations, infos)
obs = next_obs

In [30]:
rb.observations #rb里面有5组数据，或者更多，将上面执行5次

array([[[ 1.2545047e+00, -3.5584038e-03,  4.4864947e-03, ...,
         -4.7244090e-03,  2.5351311e-03,  3.8143314e-04]],

       [[ 1.2542328e+00, -2.2137435e-03,  4.1798078e-03, ...,
         -6.1492037e-02,  5.0510293e-01, -1.0451381e+00]],

       [[ 1.2534521e+00, -1.8512630e-03,  3.8029666e-03, ...,
         -3.4990493e-02,  9.2964089e-03, -1.2293504e+00]],

       ...,

       [[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00]],

       [[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00]],

       [[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
          0.0000000e+00,  0.0000000e+00,  0.0000000e+00]]], dtype=float32)

In [31]:
data = rb.sample(2)

In [32]:
data #data数据不可以直接相加

ReplayBufferSamples(observations=tensor([[ 1.2475e+00, -2.0417e-02, -1.5541e-02,  1.0309e-03, -4.0988e-02,
         -1.3609e-01, -3.5991e-01, -1.6760e+00, -1.8982e+00, -3.6956e-02,
         -1.4860e+00],
        [ 1.2521e+00, -2.7945e-03,  3.5324e-03,  2.9362e-03, -2.3318e-02,
         -4.3989e-02, -2.1110e-01, -1.7224e-01, -3.2126e-02, -2.4082e-01,
         -1.0645e+00]]), actions=tensor([[-0.0930, -0.7319, -0.1938],
        [-0.9449,  0.5070,  0.0763]]), next_observations=tensor([[ 1.2443e+00, -3.5755e-02, -3.0848e-02, -3.1008e-03, -5.3726e-02,
         -2.3152e-01, -4.3001e-01, -2.1583e+00, -1.9285e+00, -9.9490e-01,
         -1.6983e+00],
        [ 1.2501e+00, -8.5966e-03, -2.2753e-03,  1.5861e-03, -3.1309e-02,
         -1.0446e-01, -2.9022e-01, -1.2819e+00, -1.4170e+00, -1.0965e-01,
         -9.3276e-01]]), dones=tensor([[0.],
        [0.]]), rewards=tensor([[0.8156],
        [0.9251]]))

In [33]:
data2 = rb.sample(3)

In [34]:
data2

ReplayBufferSamples(observations=tensor([[ 1.2501e+00, -8.5966e-03, -2.2753e-03,  1.5861e-03, -3.1309e-02,
         -1.0446e-01, -2.9022e-01, -1.2819e+00, -1.4170e+00, -1.0965e-01,
         -9.3276e-01],
        [ 1.2545e+00, -3.5584e-03,  4.4865e-03, -1.8817e-03, -7.6674e-04,
          3.2770e-03, -9.0801e-04,  4.9594e-04, -4.7244e-03,  2.5351e-03,
          3.8143e-04],
        [ 1.2542e+00, -2.2137e-03,  4.1798e-03,  1.8088e-03, -5.0443e-03,
          2.4124e-02, -6.3062e-02,  1.5201e-01, -6.1492e-02,  5.0510e-01,
         -1.0451e+00]]), actions=tensor([[-0.3405,  0.5769, -0.3936],
        [ 0.0236,  0.9009, -0.7117],
        [ 0.8973, -0.3763, -0.1533]]), next_observations=tensor([[ 1.2475e+00, -2.0417e-02, -1.5541e-02,  1.0309e-03, -4.0988e-02,
         -1.3609e-01, -3.5991e-01, -1.6760e+00, -1.8982e+00, -3.6956e-02,
         -1.4860e+00],
        [ 1.2542e+00, -2.2137e-03,  4.1798e-03,  1.8088e-03, -5.0443e-03,
          2.4124e-02, -6.3062e-02,  1.5201e-01, -6.1492e-02,  5.0510

In [48]:
obs = torch.cat([data.observations, data2.observations])
next_obs = torch.cat([data.actions, data2.actions])
acts = torch.cat([data.next_observations, data2.next_observations])
dones = torch.cat([data.dones, data2.dones])
rews = torch.cat([data.rewards, data2.rewards])

In [52]:
from collections import namedtuple
Batch = namedtuple('ReplayBufferSamples', ['observations', 'next_observations', 'actions', 'rewards', 'dones'])
data = Batch(observations=obs, next_observations=next_obs, actions=acts, rewards=rews, dones=dones)

In [54]:
type(envs)

gymnasium.vector.sync_vector_env.SyncVectorEnv

In [55]:
type(envs.envs[0])

gymnasium.wrappers.record_episode_statistics.RecordEpisodeStatistics

In [69]:
a = [[0,1,2,3,4,5],
    [0,1,2,3,4,5]]
b = [[6,7,8,9],
    [6,7,8,9]]
c = np.concatenate([a, b],axis=1)

In [74]:
d = [1,2,3,4,5]

In [75]:
d[2:4]

[3, 4]